# 🐼 Módulo 01 · Pandas

Reconocer problemas de datos y saber qué operación de pandas los resuelve: leer, inspeccionar, filtrar, limpiar, agrupar, ordenar y combinar.

Versión web con conceptos, reconocimiento, razonamiento y proyecto: [https://rosalesluciano.github.io/datalab/](https://rosalesluciano.github.io/datalab/#/m/m01)

## Cómo trabajar este notebook
1. Ejecutá la celda ⚙️ (carga datos y verificador).
2. **Antes de escribir código**, escribí en un comentario cómo lo vas a resolver.
3. Implementá y guardá tu respuesta en `resultado` (o lo que pida el ejercicio).
4. `comprobar()` verifica. Si te trabás: `ver_pista("id", 1)`, después 2 y 3.
5. `ver_solucion("id")` es el último recurso: muestra idea → pseudocódigo → código.

## Conceptos del módulo
- **Qué problema resuelve pandas**: Trabajar con **tablas** (filas y columnas) de forma rápida y expresiva: leer, limpiar, filtrar, agrupar, combinar y resumir datos sin escribir loops.
- **Series**: Representar **una columna** de datos con etiquetas (índice): ventas por día, precios por producto.
- **Leer y guardar datos**: Traer datos desde archivos o bases de datos a un DataFrame, y exportar los resultados.
- **Inspeccionar un dataset**: Entender **qué tenés** antes de analizar: cuántas filas, qué columnas, qué tipos, qué valores raros.
- **Filtrar filas con condiciones**: Quedarte solo con las filas que cumplen una o varias condiciones.
- **Valores faltantes (NaN)**: Detectar y decidir qué hacer con datos que faltan: eliminarlos, rellenarlos o dejarlos marcados.
- **Duplicados**: Encontrar y eliminar registros repetidos que inflan conteos y totales.
- **Tipos de datos (dtypes) y conversiones**: Asegurar que cada columna tenga el tipo correcto: números como números, fechas como fechas.
- **Agrupar y agregar**: Calcular una métrica **por grupo**: promedio por categoría, total por cliente, cantidad por país.
- **Ordenar y rankings**: Ordenar filas o valores para ver los mayores/menores, armar rankings y top-N.
- **Operaciones con texto**: Limpiar y transformar columnas de texto completas: normalizar nombres, buscar palabras, quitar símbolos.
- **Fechas**: Trabajar con el tiempo: extraer mes/día, agrupar por período, calcular diferencias, series temporales.
- **Combinar tablas**: Juntar información que está en tablas distintas: pedidos con productos, clientes con pedidos; o apilar tablas con las mismas columnas.
- **Pipeline de limpieza**: Convertir datos crudos en datos confiables con una secuencia de pasos reproducible.
- **Análisis exploratorio (EDA)**: Explorar un dataset para entenderlo, encontrar problemas y generar preguntas antes de concluir.

In [ ]:
#@title ⚙️ 1) Ejecutá esta celda primero (datos y verificador) { display-mode: "form" }
"""PandasQuest - motor de corrección.

Se usa igual en el navegador (Pyodide), en los notebooks de Colab y en los tests.
Compara el `resultado` del alumno contra el de la solución oficial con tolerancia
a diferencias irrelevantes (tipos int/float, nombres de índice, dtype string, etc.).
"""
import math

import numpy as np
import pandas as pd


def _norm_s(s):
    dt = s.dtype
    if isinstance(dt, pd.CategoricalDtype) or pd.api.types.is_string_dtype(dt):
        s = s.astype(object)
    elif pd.api.types.is_extension_array_dtype(dt):
        if pd.api.types.is_numeric_dtype(dt) and not pd.api.types.is_bool_dtype(dt):
            s = s.astype("float64")
        else:
            s = s.astype(object)
    if s.dtype == object:
        s = s.where(s.notna(), None)
    return s


def _norm_index(idx):
    if isinstance(idx, pd.MultiIndex):
        return idx
    if isinstance(idx, pd.CategoricalIndex) or pd.api.types.is_string_dtype(idx.dtype):
        return pd.Index(list(idx), dtype=object)
    return idx


def _norm(o):
    if isinstance(o, pd.Series):
        o = _norm_s(o.copy())
    else:
        cols = list(o.columns)
        o = pd.DataFrame({i: _norm_s(o.iloc[:, i]) for i in range(o.shape[1])}, index=o.index)
        o.columns = _norm_index(pd.Index(cols))
    o.index = _norm_index(o.index)
    return o


def _sort(o, ignore_index):
    if ignore_index:
        o = o.reset_index(drop=True)
        try:
            if isinstance(o, pd.Series):
                o = o.sort_values(kind="mergesort")
            else:
                o = o.sort_values(by=list(o.columns), kind="mergesort")
        except TypeError:
            key = o.astype(str) if isinstance(o, pd.Series) else o.astype(str).agg("|".join, axis=1)
            o = o.iloc[np.argsort(key.to_numpy(), kind="mergesort")]
        return o.reset_index(drop=True)
    try:
        return o.sort_index()
    except TypeError:
        return o


def _tipo(x):
    nombres = {pd.DataFrame: "un DataFrame", pd.Series: "una Series", list: "una lista", dict: "un diccionario",
               tuple: "una tupla", set: "un set", str: "un texto (str)", int: "un entero (int)",
               float: "un decimal (float)", bool: "un booleano (bool)", type(None): "None (nada)",
               np.ndarray: "un array de NumPy"}
    return nombres.get(type(x), f"un {type(x).__name__}")


def _scalar_eq(u, e):
    if isinstance(e, (np.generic,)):
        e = e.item()
    if isinstance(u, (np.generic,)):
        u = u.item()
    if isinstance(e, float) or isinstance(u, float):
        try:
            u, e = float(u), float(e)
        except (TypeError, ValueError):
            return False
        if math.isnan(e):
            return math.isnan(u)
        return math.isclose(u, e, rel_tol=1e-6, abs_tol=1e-6)
    if isinstance(e, (list, tuple)) and isinstance(u, (list, tuple, np.ndarray, pd.Index)):
        u = list(u)
        return len(u) == len(e) and all(_scalar_eq(a, b) for a, b in zip(u, e))
    if isinstance(e, dict):
        return (isinstance(u, dict) and set(map(str, u)) == set(map(str, e))
                and all(_scalar_eq(u[k] if k in u else u[str(k)], v) for k, v in e.items()))
    if isinstance(e, (set, frozenset)):
        return isinstance(u, (set, frozenset)) and u == e
    try:
        return bool(u == e)
    except Exception:
        return False


def _array_compare(u, e):
    if isinstance(u, (pd.DataFrame, pd.Series)):
        u = u.to_numpy()
    try:
        ua = np.asarray(u)
    except Exception:
        return False, f"Se esperaba un array de NumPy y tu resultado es {_tipo(u)}."
    if not isinstance(u, np.ndarray) and not isinstance(u, (list, tuple)):
        return False, f"Se esperaba un array de NumPy y tu resultado es {_tipo(u)}."
    if ua.shape != e.shape:
        return False, f"Forma distinta: tu array tiene shape {ua.shape} y se esperaba {e.shape}."
    num = np.issubdtype(e.dtype, np.number) or e.dtype == bool
    try:
        if num and (np.issubdtype(ua.dtype, np.number) or ua.dtype == bool):
            ok = np.allclose(ua.astype(float), e.astype(float), rtol=1e-6, atol=1e-6, equal_nan=True)
        else:
            ok = ua.tolist() == e.tolist()
    except Exception:
        ok = False
    return (True, "") if ok else (False, "La forma es correcta pero los valores no coinciden. Revisa el cálculo.")


def pq_compare(u, e, ordered=True, ignore_index=False):
    """Devuelve (ok, mensaje)."""
    if isinstance(e, np.ndarray):
        return _array_compare(u, e)
    if isinstance(e, (pd.DataFrame, pd.Series)):
        if not isinstance(u, type(e)):
            extra = ""
            if isinstance(e, pd.Series) and isinstance(u, pd.DataFrame):
                extra = " Pista: con una sola columna usa df['col'] (corchetes simples)."
            if isinstance(e, pd.DataFrame) and isinstance(u, pd.Series):
                extra = " Pista: usa doble corchete df[['col']] o .reset_index() / .to_frame()."
            return False, f"Tu resultado es {_tipo(u)}, pero se esperaba {_tipo(e)}.{extra}"
        if isinstance(e, pd.DataFrame):
            uc, ec = [str(c) for c in u.columns], [str(c) for c in e.columns]
            if uc != ec:
                if sorted(uc) == sorted(ec):
                    return False, f"Tienes las columnas correctas pero en otro orden. Se esperaba: {ec}"
                faltan = [c for c in ec if c not in uc]
                sobran = [c for c in uc if c not in ec]
                msg = "Las columnas no coinciden."
                if faltan:
                    msg += f" Faltan: {faltan}."
                if sobran:
                    msg += f" Sobran: {sobran}."
                return False, msg
        if u.shape != e.shape:
            return False, f"Forma distinta: tu resultado tiene {u.shape} y se esperaba {e.shape} (filas, columnas)."
        u2, e2 = _norm(u), _norm(e)
        if not ordered:
            u2, e2 = _sort(u2, ignore_index), _sort(e2, ignore_index)
        elif ignore_index:
            u2, e2 = u2.reset_index(drop=True), e2.reset_index(drop=True)
        kw = dict(check_dtype=False, check_names=False, check_index_type=False,
                  check_exact=False, rtol=1e-6, atol=1e-6)
        try:
            if isinstance(e2, pd.DataFrame):
                pd.testing.assert_frame_equal(u2, e2, check_column_type=False, check_freq=False, **kw)
            else:
                pd.testing.assert_series_equal(u2, e2, check_freq=False, **kw)
        except AssertionError as err:
            txt = str(err)
            if "index" in txt.lower() and "values are different" in txt.lower() and "iloc" not in txt:
                return False, "Los valores parecen correctos pero el índice no coincide. ¿Hace falta ordenar o usar reset_index()?"
            if ordered and not ignore_index:
                try:
                    pd.testing.assert_frame_equal(
                        u2.to_frame() if isinstance(u2, pd.Series) else u2,
                        e2.to_frame() if isinstance(e2, pd.Series) else e2,
                        check_like=True, **kw)
                except AssertionError:
                    pass
                else:
                    return False, "Mismos datos, pero en otro orden. Revisa cómo ordenas las filas."
            return False, "Los valores no coinciden con lo esperado. Revisa la lógica y vuelve a intentarlo."
        return True, ""
    if isinstance(u, (pd.DataFrame, pd.Series)):
        return False, f"Tu resultado es {_tipo(u)}, pero se esperaba un valor simple ({type(e).__name__}). ¿Te falta .iloc[0], .sum() o similar?"
    if _scalar_eq(u, e):
        return True, ""
    if u is None:
        return False, ("Tu resultado es None. ¿Te falta un `return` en la función, o guardaste lo que devuelve "
                       "un método que modifica en el lugar (como `.sort()` o `.append()`)?")
    contenedores = (list, tuple, dict, set, str)
    if isinstance(e, contenedores) and not isinstance(u, type(e)) and not (
            isinstance(e, (list, tuple)) and isinstance(u, (list, tuple))):
        return False, f"Tu resultado es {_tipo(u)}, pero se esperaba {_tipo(e)}."
    if isinstance(e, (list, tuple)) and isinstance(u, (list, tuple)) and len(u) != len(e):
        return False, f"Tu resultado tiene {len(u)} elementos y se esperaban {len(e)}."
    if isinstance(e, dict) and isinstance(u, dict) and set(map(str, u)) != set(map(str, e)):
        faltan = [k for k in e if str(k) not in set(map(str, u))]
        sobran = [k for k in u if str(k) not in set(map(str, e))]
        msg = "Las claves del diccionario no coinciden."
        if faltan:
            msg += f" Faltan: {faltan}."
        if sobran:
            msg += f" Sobran: {sobran}."
        return False, msg
    txt = repr(u)
    return False, f"Tu resultado es {txt[:300] + ('…' if len(txt) > 300 else '')}, pero no es el valor esperado."


# ---------------------------------------------------------------- gráficos
def _plt():
    import matplotlib.pyplot as plt
    return plt


def _fig():
    plt = _plt()
    return plt.gcf() if plt.get_fignums() else None


def _ax(i=0):
    f = _fig()
    if f is None or len(f.axes) <= i:
        raise LookupError("no hay ningún gráfico dibujado")
    return f.axes[i]


def _legend(ax=None):
    ax = ax or _ax()
    lg = ax.get_legend()
    return [t.get_text() for t in lg.get_texts()] if lg else []


def _heights(ax=None):
    return [round(float(p.get_height()), 6) for p in (ax or _ax()).patches]


def _widths(ax=None):
    return [round(float(p.get_width()), 6) for p in (ax or _ax()).patches]


def _xticks(ax=None):
    return [t.get_text() for t in (ax or _ax()).get_xticklabels()]


def _hex(color):
    import matplotlib.colors as mcolors
    return mcolors.to_hex(color)


_HELPERS = {"_fig": _fig, "_ax": _ax, "_legend": _legend, "_heights": _heights,
            "_widths": _widths, "_xticks": _xticks, "_hex": _hex}


def pq_prepare():
    """Deja matplotlib limpio antes de ejecutar código (si está cargado)."""
    import sys
    if "matplotlib.pyplot" in sys.modules:
        plt = sys.modules["matplotlib.pyplot"]
        plt.close("all")
        plt.show = lambda *a, **k: None


def pq_check(ns, get_expected, opts):
    """Evalúa el ejercicio.

    ns = namespace del alumno. get_expected = función que devuelve el namespace de la
    solución (solo se ejecuta si hace falta, para no pisar los gráficos del alumno).
    """
    custom = opts.get("custom")
    if custom:
        if opts.get("plot") and _fig() is None:
            return False, "Todavía no hay ningún gráfico. ¡Dibuja algo! 🎨"
        env = dict(_HELPERS)
        env.update(ns)
        try:
            ok = bool(eval(custom, env))
        except Exception as err:
            return False, f"{opts.get('custom_msg', 'Todavía no es correcto.')} ({type(err).__name__}: {err})"
        return ok, "" if ok else opts.get("custom_msg", "Todavía no es correcto. ¡Sigue intentándolo!")
    if "resultado" not in ns:
        return False, "Guarda tu respuesta en la variable `resultado`."
    exp_ns = get_expected() if callable(get_expected) else get_expected
    return pq_compare(ns["resultado"], exp_ns["resultado"],
                      ordered=opts.get("ordered", True),
                      ignore_index=opts.get("ignore_index", False))


_ERR_TIPS = [
    ("truth value of a Series is ambiguous", "Para combinar condiciones usá & (y), | (o) y poné cada condición entre paréntesis."),
    ("can't multiply sequence", "Estás multiplicando un texto (str). ¿El número llegó como texto? Convertilo con int() o float()."),
    ("can only concatenate str", "Estás sumando texto con otra cosa. Convertí con str() para unir textos, o con float() para hacer cuentas."),
    ("unsupported operand type", "Los dos lados de la operación tienen tipos incompatibles (por ejemplo, int + str o None + float). Revisá type() de cada valor."),
    ("could not convert string to float", "Hay un texto que no representa un número (por ejemplo 'n/d' o ''). Filtralo o capturá el ValueError con try/except."),
    ("invalid literal for int", "Hay un texto que no representa un entero. Revisá el valor o usá try/except ValueError."),
    ("index out of range", "Pediste una posición que no existe. Recordá: las posiciones van de 0 a len()-1, y -1 es el último."),
    ("values to unpack", "La cantidad de variables a la izquierda no coincide con la cantidad de valores que llegan."),
    ("ZeroDivisionError", "Dividiste por cero. ¿La lista o el divisor pueden estar vacíos? Verificalo antes de dividir."),
    ("'NoneType' object", "Un valor es None. Suele pasar con una función sin return o con métodos como .sort() que devuelven None."),
    ("KeyError", "Esa clave o columna no existe. Revisá mayúsculas, tildes y espacios; con dicts podés usar .get()."),
    ("NameError", "Usás una variable que no existe (todavía). ¿Error de tipeo o la definiste más abajo?"),
    ("IndentationError", "Revisá la sangría: el contenido de if, for, def, etc. va con 4 espacios."),
    ("SyntaxError", "Error de sintaxis: revisá paréntesis, comillas, dos puntos (:) al final de if/for/def y que cada línea esté completa."),
    ("AttributeError", "Ese método o atributo no existe para este tipo de objeto. Revisá el nombre y qué tipo tiene la variable."),
    ("MergeError", "Revisá las columnas que usás para unir (on / left_on / right_on)."),
    ("TypeError", "Algún tipo de dato no encaja con la operación. Revisá type() de las variables involucradas."),
    ("ValueError", "El tipo es correcto pero el valor no es válido para esta operación."),
]


def pq_error_tip(msg):
    for key, tip in _ERR_TIPS:
        if key in msg:
            return tip
    return ""


_SETUP = 'import pandas as pd\nimport numpy as np\n\nclientes = pd.DataFrame({\n    "cliente_id": range(1, 13),\n    "nombre": ["Ana", "Bruno", "Carla", "Diego", "Elena", "Facundo",\n               "Gabriela", "Hugo", "Inés", "Julián", "Karina", "Lucas"],\n    "ciudad": ["Madrid", "Buenos Aires", "CDMX", "Bogotá", "Madrid", "Lima",\n               "Buenos Aires", "CDMX", "Santiago", "Bogotá", "Madrid", "Lima"],\n    "pais": ["España", "Argentina", "México", "Colombia", "España", "Perú",\n             "Argentina", "México", "Chile", "Colombia", "España", "Perú"],\n    "edad": [34, 28, 45, 23, 51, 39, 30, 27, 42, 36, 25, 48],\n    "segmento": ["Premium", "Básico", "Premium", "Básico", "Premium", "Estándar",\n                 "Estándar", "Básico", "Premium", "Estándar", "Básico", "Estándar"],\n    "email": ["ana@mail.com", None, "carla@mail.com", "diego@mail.com", None, "facu@mail.com",\n              "gabi@mail.com", None, "ines@mail.com", "julian@mail.com", "kari@mail.com", "lucas@mail.com"],\n    "fecha_registro": pd.to_datetime([\n        "2023-01-15", "2023-02-03", "2023-02-20", "2023-03-11", "2023-04-02", "2023-05-19",\n        "2023-06-07", "2023-07-23", "2023-08-30", "2023-09-14", "2023-10-05", "2023-11-28"]),\n})\n\nproductos = pd.DataFrame({\n    "producto_id": range(1, 11),\n    "producto": ["Laptop", "Mouse", "Teclado", "Monitor", "Auriculares",\n                 "Webcam", "Silla gamer", "Escritorio", "Tablet", "Micrófono"],\n    "categoria": ["Computación", "Accesorios", "Accesorios", "Computación", "Audio",\n                  "Accesorios", "Muebles", "Muebles", "Computación", "Audio"],\n    "precio": [1200.0, 25.5, 45.0, 300.0, 80.0, 60.0, 250.0, 180.0, 450.0, 95.0],\n    "stock": [5, 150, 80, 20, 60, 0, 12, 7, 15, 0],\n})\n\n_estados = ["entregado", "enviado", "entregado", "cancelado", "entregado", "pendiente"]\npedidos = pd.DataFrame({\n    "pedido_id": range(1001, 1031),\n    "cliente_id": [(i * 7) % 11 + 1 for i in range(29)] + [99],\n    "producto_id": [(i * 3) % 10 + 1 for i in range(30)],\n    "cantidad": [(i % 4) + 1 for i in range(30)],\n    "fecha": pd.date_range("2024-01-03", periods=30, freq="5D"),\n    "estado": [_estados[i % 6] for i in range(30)],\n})\n\nempleados = pd.DataFrame({\n    "emp_id": range(1, 11),\n    "nombre": ["Marta", "Pablo", "Sofía", "Tomás", "Valeria",\n               "Andrés", "Lucía", "Martín", "Paula", "Ramiro"],\n    "depto": ["Dirección", "Ventas", "Ventas", "Ventas", "IT",\n              "IT", "IT", "Marketing", "Marketing", "Ventas"],\n    "salario": [9000, 4200, 3900, 4000, 5200, 4800, 6100, 3700, 4100, 3500],\n    "jefe_id": pd.array([None, 1, 2, 2, 1, 5, 5, 1, 8, 2], dtype="Int64"),\n    "fecha_ingreso": pd.to_datetime([\n        "2015-03-01", "2018-06-15", "2019-01-10", "2020-09-01", "2017-11-20",\n        "2021-02-01", "2016-08-08", "2022-04-18", "2019-07-07", "2023-01-09"]),\n})\n\n_i = np.arange(90)\n_f = pd.date_range("2024-01-01", periods=90, freq="D")\n_v = 1000 + 8 * _i + 250 * (_f.dayofweek >= 5) + ((_i * 37) % 11) * 20\n_v = np.where(np.isin(_i, [20, 55, 77]), _v + 1500, _v)\nventas_diarias = pd.DataFrame({\n    "fecha": _f,\n    "ventas": _v.astype(float),\n    "visitas": (300 + (_i * 13) % 97 + 2 * _i).astype(int),\n})\n\nsucio = pd.DataFrame({\n    "id": [1, 2, 2, 3, 4, 5, 6, 7],\n    "nombre": ["  ana ", "BRUNO", "BRUNO", "carla", None, "Diego  ", "elena", "facundo"],\n    "edad": ["34", "28", "28", "n/d", "23", "51", None, "39"],\n    "ciudad": ["madrid", "Buenos Aires", "Buenos Aires", " santiago", "bogotá", "Madrid ", "madrid", "LIMA"],\n    "monto": ["$1,200.50", "$300", "$300", "$45.00", None, "$80.5", "$99", "$1,000"],\n})\ndel _i, _f, _v, _estados\n'
_SOL_B64 = '...'
_TABLAS = ['clientes', 'productos', 'pedidos', 'empleados', 'ventas_diarias', 'sucio']

import base64 as _b64, json as _json, re as _re, warnings as _w
_w.simplefilter("ignore")
_SOL = _json.loads(_b64.b64decode(_SOL_B64).decode("utf-8"))

def reiniciar_datos():
    """Vuelve a cargar todos los datos limpios (cada ejercicio empieza de cero)."""
    globals().pop("resultado", None)
    exec(_SETUP, globals())
    pq_prepare()

def _codigo_celda():
    try:
        src = get_ipython().history_manager.input_hist_raw[-1]
    except Exception:
        return ""
    src = "\n".join(l for l in src.splitlines() if "comprobar(" not in l)
    return "\n".join(_re.sub(r"#.*$", "", l) for l in src.splitlines())

def comprobar(ej):
    info = _SOL[ej]
    def esperado():
        exp_ns = {}
        exec(_SETUP, exp_ns)
        exec(info["solution"], exp_ns)
        return exp_ns
    g = globals()
    g["_pq_code"] = _codigo_celda()
    ok, msg = pq_check(g, esperado, info["check"])
    from IPython.display import HTML, display
    import html as _h
    if ok:
        display(HTML('<div style="padding:12px 16px;border-radius:12px;background:#0f9d6e;color:#fff;'
                     'font:600 15px system-ui">✅ Verificado. Ahora explicá con tus palabras por qué funciona '
                     'y probá una variante.</div>'))
    else:
        display(HTML('<div style="padding:12px 16px;border-radius:12px;background:#b4235a;color:#fff;'
                     'font:500 15px system-ui">Todavía no. ' + _h.escape(msg) + '</div>'))
    if "resultado" in g:
        return g["resultado"]

def ver_pista(ej, nivel=1):
    """Pistas progresivas (1-3). Pedí la siguiente solo si la anterior no alcanzó."""
    pistas = _SOL[ej]["hints"]
    print(f"Pista {nivel}/{len(pistas)}: {pistas[min(nivel, len(pistas)) - 1]}")

def ver_solucion(ej):
    """Último recurso: primero la idea, después el pseudocódigo y al final el código."""
    info = _SOL[ej]
    print("💡 Idea:\n" + info["explain"] + "\n")
    print("🧭 Pseudocódigo:\n" + info["pseudo"] + "\n")
    print("🧑‍💻 Código:\n" + info["solution"])

reiniciar_datos()
print("✅ Datos cargados:", ", ".join(_TABLAS))


---
# 📌 Qué resuelve pandas

## 1. Hola, pandas 👋
`Código` · `Muy fácil`

Creá una Series con los valores `10, 20, 30` y las etiquetas `'a', 'b', 'c'`. Guardala en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

# pandas ya está importado como pd
# Escribe tu código aquí 👇
resultado = 

comprobar("pd01")

## 2. Series desde un diccionario
`Código` · `Muy fácil`

Tenés las ventas de tres días en el diccionario `ventas`. Convertilo en una Series (cada día como etiqueta). Guardala en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

ventas = {'Lun': 120, 'Mar': 95, 'Mié': 143}
resultado = 

comprobar("pd02")

## 3. Cálculos sin bucles
`Código` · `Fácil`

Los precios de `precios` no incluyen IVA. Calculá los precios con un **IVA del 21%** y guardá la Series en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

precios = pd.Series([100, 250, 80, 40], index=['Mouse', 'Teclado', 'Webcam', 'Cable'])
resultado = 

comprobar("pd04")

## 4. Estadística exprés
`Código` · `Fácil`

¿Cuál es la **nota media** del grupo? Calculala a partir de `notas` y guardala en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

notas = pd.Series([7, 9, 6, 8, 10, 5], index=['Ana', 'Bruno', 'Carla', 'Diego', 'Elena', 'Facu'])
resultado = 

comprobar("pd06")

---
# 📌 Leer e inspeccionar datos

## 5. Leer un CSV
`Código` · `Medio`

El texto `csv` contiene datos de ciudades en formato CSV. Convertilo en un DataFrame (usá `io.StringIO` para tratarlo como archivo) y guardalo en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

import io

csv = '''ciudad,habitantes,pais
Madrid,3300000,España
Lima,10000000,Perú
Bogotá,7900000,Colombia'''

resultado = 

comprobar("pd18")

## 6. Un vistazo rápido
`Código` · `Muy fácil`

Quiero echar un vistazo a la tabla `productos`: guardá en `resultado` sus **3 primeras filas**.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("pd14")

## 7. Resumen estadístico
`Código` · `Muy fácil`

Antes de analizar `productos`, querés ver de un vistazo la media, mínimo, máximo y percentiles de sus columnas numéricas. Guardá ese resumen en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("pd16")

## 8. ¿Cuántos de cada uno?
`Código` · `Muy fácil`

¿Cuántos clientes hay de cada `pais`? Guardá la Series con el conteo en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("pd17")

## 9. Exportar resultados
`Código` · `Medio`

Tenés que enviar `productos` como texto CSV **sin el índice** (para que al reabrirlo no aparezca una columna `Unnamed: 0`). Guardá ese texto en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("pd19")

---
# 📌 Filtrar con condiciones

## 10. Filtrar valores
`Código` · `Medio`

De `precios`, quedate solo con los productos que cuestan **más de 90**. Guardá el resultado.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

precios = pd.Series([100, 250, 80, 40], index=['Mouse', 'Teclado', 'Webcam', 'Cable'])
resultado = 

comprobar("pd05")

## 11. Una condición
`Código` · `Fácil`

Marketing quiere los clientes **mayores de 40 años**. Guardalos en `resultado` (todas las columnas).

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("df05")

## 12. Combinar condiciones
`Código` · `Medio`

Necesitás los clientes de **España** que además sean del segmento **Premium**.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("df06")

## 13. Transferencia: España o Premium
`Transferencia` · `Medio`

Guardá en `resultado` los clientes que sean de **España** **o** del segmento **Premium** (o ambos).

> 🔁 **Qué cambia:** Antes las dos condiciones debían cumplirse a la vez (**y**). Ahora alcanza con que se cumpla **una** de las dos (**o**).

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("pd-g01")

## 14. Varios valores con isin
`Código` · `Medio`

Logística necesita los pedidos que todavía están en camino: los de estado **enviado** o **pendiente**.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("df07")

## 15. query(): filtros legibles
`Código` · `Medio`

¿Qué productos baratos (**precio menor a 100**) tenemos **con stock** (stock mayor a 0)? Resolvelo con `query` para practicar la forma legible.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = productos.query("")

comprobar("df08")

## 16. Filas con datos faltantes
`Código` · `Medio`

Para una campaña hay que contactar por otro medio a los clientes que **no tienen email** cargado. Guardalos en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("df09")

## 17. El filtro ambiguo
`Debugging` · `Fácil`

Este filtro de clientes argentinos mayores de 25 años da un error raro. Encontrá y corregí el problema.

> 🐞 El código tiene un error: ejecutalo, leé el mensaje, explicá la causa en un comentario y corregilo.

In [ ]:
reiniciar_datos()  # datos limpios

resultado = clientes[clientes['pais'] == 'Argentina' and clientes['edad'] > 25]

comprobar("pd-e01")

---
# 📌 Datos sucios: faltantes, duplicados y tipos

## 18. Datos que faltan
`Código` · `Medio`

El sensor `temperaturas` tiene lecturas faltantes. ¿**Cuántas** faltan? Guardá ese número en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

temperaturas = pd.Series([21.5, None, 19.8, 25.1, None, 22.4, None])
resultado = 

comprobar("pd08")

## 19. Rellenar y descartar nulos
`Código` · `Medio`

En `sucio`, una fila sin `nombre` o sin `monto` no sirve para el reporte. Eliminá esas filas (pero no las que solo tienen otros datos faltantes).

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("df16")

## 20. Adiós, duplicados
`Código` · `Muy fácil`

La tabla `sucio` tiene un registro cargado dos veces. Eliminá las filas duplicadas y guardá el resultado.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("df14")

## 21. Texto a número
`Código` · `Medio`

La columna `monto` de `sucio` tiene valores como `'$1,200.50'` y no se puede sumar. Convertila a número decimal (los nulos deben seguir siendo nulos) y guardá la Series en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("df15")

## 22. La conversión que no convierte
`Debugging` · `Fácil`

Este código intenta convertir los montos de `sucio` a número, pero falla. Corregilo para obtener la Series numérica.

> 🐞 El código tiene un error: ejecutalo, leé el mensaje, explicá la causa en un comentario y corregilo.

In [ ]:
reiniciar_datos()  # datos limpios

resultado = sucio['monto'].astype(float)

comprobar("pd-e02")

## 23. Limpiar texto con .str
`Código` · `Medio`

Los nombres de `nombres` llegan con espacios de más y mayúsculas mezcladas. Normalizalos (sin espacios sobrantes, formato `Ana`, `Bruno`…).

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

nombres = pd.Series(['  ana ', 'BRUNO', 'carla  ', 'dIEGO'])
resultado = 

comprobar("pd11")

## 24. Pipeline de limpieza
`Código` · `Difícil`

Limpia `sucio` en una sola cadena:
1. Elimina los duplicados.
2. `ciudad`: sin espacios sobrantes y en formato Título (`Madrid`, `Lima`...).
3. `edad`: numérica (los valores inválidos pasan a NaN).

Guarda el DataFrame resultante, con las mismas columnas.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = (
    sucio
    # .drop_duplicates()
    # .assign(...)
)

comprobar("df17")

---
# 📌 Agrupar y agregar

## 25. Implementar el plan
`Código` · `Medio`

Implementá el plan anterior: el **promedio de unidades** (`cantidad`) por pedido **para cada categoría**. Guardá la Series en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("pd-d01")

## 26. Media por grupo
`Código` · `Medio`

¿Cuál es la **edad promedio** de los clientes de cada `segmento`?

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("df18")

## 27. Transferencia: el máximo por categoría
`Transferencia` · `Fácil`

Obtené el **precio máximo** de cada `categoria` de `productos` y guardá la Series en `resultado`.

> 🔁 **Qué cambia:** Antes calculaste el **promedio** de unidades por categoría. Ahora la pregunta es el **máximo precio** por categoría (solo con la tabla productos).

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("pd-g02")

## 28. Varias métricas a la vez
`Código` · `Difícil`

Para un informe por `categoria` de `productos`, necesitás tres columnas: `precio_medio` (media de precio), `stock_total` (suma de stock) y `n_productos` (cuántos productos hay).

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = productos.groupby('categoria').agg(
    
)

comprobar("df19")

## 29. Filtrar grupos (HAVING)
`Código` · `Difícil`

Queremos saber en qué países tenemos una base de clientes: guardá los países con **al menos 2** clientes (una Series con el conteo de cada país).

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

conteo = clientes['pais'].value_counts()
resultado = 

comprobar("df20")

## 30. La IA resume ventas por estado
`IA copiloto` · `Medio`

Pediste a una IA 'la facturación total por estado de pedido'. Te dio este código. Antes de confiar en el número, analizalo.

> 🤖 Prompt usado:
> Tengo dos DataFrames: pedidos (producto_id, cantidad, estado) y productos (producto_id, precio). Quiero la facturación (cantidad × precio) total por estado. Explicame el plan y cada línea.

Antes de ejecutarlo, explicá en comentarios qué hace cada línea.

In [ ]:
reiniciar_datos()  # datos limpios

df = pedidos.merge(productos, on='producto_id')
df['importe'] = df['cantidad'] * df['precio']
resultado = df.groupby('estado')['importe'].sum().sort_values(ascending=False)

comprobar("pd-f01")

## 31. El conteo que no es conteo
`Debugging` · `Medio`

Querés **cuántos pedidos** hubo en cada `estado`, pero el resultado da números demasiado grandes. Corregilo.

> 🐞 El código tiene un error: ejecutalo, leé el mensaje, explicá la causa en un comentario y corregilo.

In [ ]:
reiniciar_datos()  # datos limpios

resultado = pedidos.groupby('estado')['cantidad'].sum()

comprobar("pd-e03")

---
# 📌 Ordenar y rankings

## 32. Ordenar un ranking
`Código` · `Fácil`

Armá el **ranking** de `notas`: de la más alta a la más baja.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

notas = pd.Series([7, 9, 6, 8, 10, 5], index=['Ana', 'Bruno', 'Carla', 'Diego', 'Elena', 'Facu'])
resultado = 

comprobar("pd07")

## 33. Ordenar con desempate
`Código` · `Medio`

Ordená `clientes` por `pais` (A→Z) y, **dentro de cada país**, por `edad` de mayor a menor.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("df10")

---
# 📌 Fechas

## 34. Trabajar con fechas
`Código` · `Medio`

¿Cuántos pedidos hubo en cada **mes** (número del 1 al 12)? Guardá la Series en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("df28")

## 35. Series temporales
`Código` · `Difícil`

Dirección quiere ver las ventas de `ventas_diarias` **por semana** (`'W'`). Guardá la Series con el total semanal en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("df29")

---
# 📌 Combinar tablas

## 36. Inner join con merge
`Código` · `Medio`

Para analizar los pedidos necesitás ver, en cada pedido, los datos del producto. Uní `pedidos` con `productos`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("df22")

## 37. ¿Cuánto facturamos?
`Código` · `Medio`

¿Cuánto facturamos? Calculá la facturación total (`cantidad × precio`) de los pedidos que **no** están cancelados. Guardá un número.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

df = pedidos.merge(productos, on='producto_id')
resultado = 

comprobar("df23")

## 38. Clientes sin pedidos
`Código` · `Difícil`

¿Qué clientes **nunca** hicieron un pedido? Guardalos en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("df24")

## 39. Apilar tablas
`Código` · `Medio`

Armá una lista con los clientes de **España** primero y después los de **Perú**, con el índice renumerado desde 0.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

espana = clientes[clientes['pais'] == 'España']
peru = clientes[clientes['pais'] == 'Perú']
resultado = 

comprobar("df25")

## 40. El merge que no encuentra la clave
`Debugging` · `Fácil`

Este código intenta traer el nombre del cliente a cada pedido, pero falla. Corregilo.

> 🐞 El código tiene un error: ejecutalo, leé el mensaje, explicá la causa en un comentario y corregilo.

In [ ]:
reiniciar_datos()  # datos limpios

resultado = pedidos.merge(clientes, on='id')

comprobar("pd-e04")

---
# 📌 Proyecto: análisis de ventas de la tienda

## 41. Fase 2 · Facturación por categoría
`Código` · `Difícil`

Calculá la **facturación por categoría** (sin pedidos cancelados), ordenada de mayor a menor. Guardá la Series en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("pd-p2")

## 42. Fase 3 · Top 3 clientes
`Código` · `Difícil`

Obtené los **3 clientes** (por `nombre`) que más facturaron, sin contar pedidos cancelados. Guardá una Series `nombre → facturación` con los 3 mayores, de mayor a menor.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("pd-p3")

## 43. Fase 4 · El importe desalineado
`Debugging` · `Difícil`

Un compañero calculó el importe sin hacer merge y la facturación total no coincide con la tuya. Encontrá el error.

> 🐞 El código tiene un error: ejecutalo, leé el mensaje, explicá la causa en un comentario y corregilo.

In [ ]:
reiniciar_datos()  # datos limpios

pedidos['importe'] = pedidos['cantidad'] * productos['precio']
resultado = pedidos['importe'].sum()

comprobar("pd-p4")

---
# 🏁 Fin del notebook
Volvé a la [plataforma](https://rosalesluciano.github.io/datalab/) para los ejercicios de reconocimiento, razonamiento y el modo **¿Sé esto?**.